## Download The Datasets


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="YclYQ8H6e4HGkMqq5V8G")
# project = rf.workspace("pylab-uuz6i").project("calculator labeling")
# Change "calculator labeling" to "calculator-labeling"
project = rf.workspace("pylab-uuz6i").project("calculator-labeling")
dataset = project.version(1).download("coco")

In [ ]:
import os
import torch
import torchvision
from pycocotools.coco import COCO
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt 


## Prepare Dataset Class !

In [ ]:


class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, annotation_file):
        self.root_dir = root_dir
        self.coco = COCO(annotation_file)

        # SMART FIX: if images are not labels then skip the image
        all_ids = list(sorted(self.coco.imgs.keys()))
        self.ids = [img_id for img_id in all_ids if len(self.coco.getAnnIds(imgIds=img_id)) > 0]

        # Debugging 
        print(f"Total images found: {len(all_ids)}")
        print(f"Valid images with labels: {len(self.ids)}")
        if len(all_ids) != len(self.ids):
            print(f"Ignored {len(all_ids) - len(self.ids)} image(s) because they had no labels.")

    def __getitem__(self, index):
        img_id = self.ids[index]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        coco_annotation = self.coco.loadAnns(ann_ids)

        # Load Image
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.root_dir, img_info['file_name'])
        img = Image.open(img_path).convert("RGB")
        img_tensor = torchvision.transforms.functional.to_tensor(img)

        # Load Boxes and Masks
        num_objs = len(coco_annotation)
        boxes, masks = [], []

        for i in range(num_objs):
            xmin, ymin, width, height = coco_annotation[i]['bbox']
            boxes.append([xmin, ymin, xmin + width, ymin + height])
            masks.append(self.coco.annToMask(coco_annotation[i]))

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.ones((num_objs,), dtype=torch.int64), # 1 = Calculator
            "masks": torch.as_tensor(np.array(masks), dtype=torch.uint8)
        }
        return img_tensor, target

    def __len__(self):
        return len(self.ids)

print("Step 1 Done:  Dataset Class is Ready!")

In [ ]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def get_simple_model(num_classes):
    # 1. Bana banaya standard model load karein
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

    # 2. Bounding Box detector ko 2 classes (Background + Calculator) par set karein
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # 3. Mask detector ko 2 classes par set karein
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)

    return model

print("Step 2 Done: Model Architecture Ready!")

## Load Datasets

In [ ]:
# Setup Paths (Make sure this path matches exactly where your data is in Colab)
DATASET_DIR = "../datasets/calculator-labeling-1/train"
JSON_FILE = os.path.join(DATASET_DIR, "_annotations.coco.json")

# Prepare Data
dataset = SimpleDataset(DATASET_DIR, JSON_FILE)
data_loader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))


In [ ]:


# Prepare Model & GPU
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = get_simple_model(num_classes=2)
model.to(device)

# Set Optimizer (Learning Rate)
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)

# Start Training Loop
model.train()
num_epochs = 10

# <--- NEW: Loss save karne ke liye list --->
epoch_losses = []

print("Training Started...\n")
for epoch in range(num_epochs):
    epoch_loss = 0

    for images, targets in data_loader:
        # Move images and targets to GPU
        images = list(img.to(device) for img in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Calculate Loss
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Optimize
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

    # <--- NEW: Har epoch ka average loss calculate aur save karein --->
    avg_loss = epoch_loss / len(data_loader)
    epoch_losses.append(avg_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] - Average Loss: {avg_loss:.4f}")

# Save the finished model
torch.save(model.state_dict(), 'simple_calculator_model.pth')
print("\nTraining Complete! Model saved as 'simple_calculator_model.pth'")



In [ ]:
# =======================================================
# <---  GRAPH GENERATION --->
# =======================================================
print("\nGenerating Training Loss Graph...")
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), epoch_losses, marker='o', linestyle='-', color='red', linewidth=2, markersize=8)

# Styling
plt.title("Mask R-CNN Training Loss", fontsize=16, fontweight='bold')
plt.xlabel("Epochs", fontsize=14)
plt.ylabel("Average Loss", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(range(1, num_epochs + 1))

# Save & Show
output_graph_path = "/content/training_loss_curve.png"
plt.savefig(output_graph_path, dpi=300, bbox_inches='tight')
print(f"Graph successfully saved at: {output_graph_path}")
plt.show()